# 24.3 设计信息流 / Design a News Feed (Facebook / Twitter / 小红书)

**中文**:信息流(feed / 时间线)是社交产品的心脏——Facebook、Twitter、抖音、小红书,你刷的那条无穷无尽的内容流就是它。设计信息流是经典系统设计题,它和推荐(24.1)共享排序漏斗,但有两个鲜明的自身特色:①**多目标价值模型**——feed 不只优化"会不会点",而要综合"会点赞吗?会评论吗?会转发吗?会看多久?"等多个预测,融合成一个排序分;②**多样性重排(必须)**——纯按分数排,你的 feed 会被一个高产的大 V 刷屏,体验极差,所以要用 **MMR** 之类做多样性。还有一个纯工程但极高频的架构问题:**推模式 vs 拉模式(fan-out on write vs read)**——怎么在亿级用户下高效地"生成每个人的 feed"。本节从零实现多目标价值融合和 MMR 多样性重排,并讲清 feed 系统设计的完整框架。
**English**: The feed (timeline) is the heart of social products — Facebook, Twitter, TikTok, Xiaohongshu; that endless stream of content you scroll is it. Designing a feed is a classic system-design question. It shares the ranking funnel with recommendation (24.1) but has two distinctive traits: ① **a multi-objective value model** — the feed optimizes not just "will they click" but combines multiple predictions ("will they like? comment? share? how long will they watch?") into one ranking score; ② **diversity re-ranking (mandatory)** — pure score-order lets one prolific influencer flood your feed, a terrible experience, so use **MMR**-like diversity. There's also a pure-engineering but very frequent architecture question: **push vs pull (fan-out on write vs read)** — how to efficiently "build each person's feed" at billion-user scale. This section implements multi-objective value fusion and MMR diversity re-ranking from scratch, and clarifies the complete feed-system-design framework.

---

**中文**:**信息流的两个独特设计**:
**English**: **Two distinctive designs of the feed**:
- **中文**:**多目标价值模型(multi-objective value model)**:feed 的排序目标不是单一的。如果只优化点击率,会滋生标题党;只优化观看时长,会推低质上瘾内容。所以现代 feed 预测**多个行为的概率**(点赞、评论、转发、观看时长、关注、举报……),再用一个**价值函数**加权融合成排序分:$\text{score}=w_1 P(\text{like})+w_2 P(\text{comment})+w_3 P(\text{share})+\dots-w_k P(\text{report})$。权重 $w$ 编码了产品的价值观(鼓励什么、惩罚什么)。多目标模型常用 **MMoE / ESMM(接 20.11)** 共享底层、多个输出头。
  **Multi-objective value model**: the feed's ranking objective isn't singular. Optimizing only click-through breeds clickbait; only watch time pushes low-quality addictive content. So modern feeds predict **probabilities of multiple actions** (like, comment, share, watch time, follow, report…), then fuse them into a ranking score with a **value function**: $\text{score}=w_1 P(\text{like})+w_2 P(\text{comment})+w_3 P(\text{share})+\dots-w_k P(\text{report})$. The weights $w$ encode the product's values (what to encourage, what to penalize). Multi-objective models often use **MMoE / ESMM (per 20.11)** with a shared bottom and multiple heads.
- **中文**:**多样性重排(diversity re-ranking)**:纯按价值分排序,会让一个高产/高分的作者、或一个热门话题**霸屏**——用户体验单一、疲劳、流失。必须在重排阶段注入多样性:**MMR(Maximal Marginal Relevance)** 在"相关性"和"与已选内容的差异"之间权衡,让 feed 兼顾好看和多元(不同作者、话题、内容类型)。还要考虑新鲜度、去重、不连续同类。
  **Diversity re-ranking**: pure value-score order lets one prolific/high-scoring author or one hot topic **dominate the screen** — monotonous, fatiguing, causing churn. Diversity must be injected in re-ranking: **MMR (Maximal Marginal Relevance)** trades off "relevance" against "difference from already-selected content," making the feed both engaging and diverse (varied authors, topics, content types). Also consider freshness, dedup, no consecutive same-type.

**中文**:**架构:推模式 vs 拉模式(fan-out,纯工程但高频考)**:
**English**: **Architecture: push vs pull (fan-out, pure engineering but frequently asked)**:
- **中文**:**推模式(fan-out on write)**:用户发帖时,**立即把这条帖子推到所有粉丝的 feed 收件箱**(预计算)。读 feed 时直接取,**读快**;但大 V 发一条要写给几千万粉丝,**写爆炸**。
  **Push (fan-out on write)**: when a user posts, **immediately push it to all followers' feed inboxes** (precompute). Reading the feed just fetches, so **reads are fast**; but a celebrity's one post writes to tens of millions of followers, a **write explosion**.
- **中文**:**拉模式(fan-out on read)**:发帖只存自己的时间线。用户刷 feed 时,**实时拉取所有关注对象的帖子再合并排序**。**写便宜**;但读时要拉合并很多人,**读慢**。
  **Pull (fan-out on read)**: posting only stores to one's own timeline. When reading, **fetch and merge posts from all followed accounts in real time**. **Writes are cheap**; but reads must pull and merge many accounts, so **reads are slow**.
- **中文**:**混合(实践方案)**:普通用户用推模式(粉丝少,写便宜、读快);大 V 用拉模式(避免写爆炸)——读 feed 时把"推来的普通内容"和"实时拉的大 V 内容"合并。这是 Twitter 等的经典解法。
  **Hybrid (the practical solution)**: ordinary users use push (few followers, cheap writes, fast reads); celebrities use pull (avoid write explosion) — when reading, merge "pushed ordinary content" with "pulled celebrity content." This is the classic Twitter-style solution.

> 💡 **面试速查 / Interview cheat-sheet（★★★ feed 系统设计, 高频）**
> **中文**:**Feed 系统**:共享推荐漏斗(候选生成→排序→重排), 两大特色:①**多目标价值模型**——预测多个行为概率(点赞/评论/转发/时长/关注/举报), 价值函数加权融合成排序分(权重编码产品价值观, 防标题党/上瘾), 用 MMoE/ESMM(20.11);②**多样性重排必须**——纯分数排会被大 V/热门话题刷屏, 用 **MMR** 权衡相关性与多样性(作者/话题/类型), 加新鲜度/去重。**架构:fan-out**:①**推(write)**预计算写入粉丝收件箱→读快写爆炸(大V灾难)②**拉(read)**读时实时合并关注者→写便宜读慢③**混合**(普通用户推+大V拉, 读时合并, 工业标准)。**候选来源**:关注/好友 + 推荐(涨活跃)+ 广告。**关键难题**:多目标权重调优(价值观权衡)、多样性vs相关性、大V写放大、实时性(新鲜内容)、**回音室/成瘾**的伦理问题、冷启动。**评估**:在线 A/B——留存/时长/互动 + 护栏(不推有害、多样性、长期留存 vs 短期点击)。面试金句:*"设计 feed:候选(关注+推荐)→多目标价值模型(预测点赞/评论/时长等, 价值函数加权融合, 用 MMoE)→多样性重排(MMR 防大V刷屏)+新鲜度; 架构用 fan-out 混合模式(普通用户推、大V拉、读时合并)解决大V写爆炸; 关键是多目标权重编码价值观防标题党上瘾、多样性与相关性权衡, 靠在线 A/B 看长期留存而非短期点击。"*
> **English**: **Feed system**: shares the recommendation funnel (candidate generation → ranking → re-ranking), two distinctive traits: ① **multi-objective value model** — predict multiple action probabilities (like/comment/share/dwell/follow/report), fuse into a ranking score with a value function (weights encode product values, prevent clickbait/addiction), using MMoE/ESMM (20.11); ② **diversity re-ranking is mandatory** — pure score-order gets flooded by celebrities/hot topics, use **MMR** to trade off relevance vs diversity (author/topic/type), plus freshness/dedup. **Architecture: fan-out**: ① **push (write)** precompute into follower inboxes → fast reads, write explosion (celebrity disaster) ② **pull (read)** merge followees at read time → cheap writes, slow reads ③ **hybrid** (ordinary users push + celebrities pull, merge at read, the industry standard). **Candidate sources**: follows/friends + recommendation (boost activity) + ads. **Key challenges**: multi-objective weight tuning (value tradeoffs), diversity vs relevance, celebrity write amplification, freshness, the ethics of **echo chambers/addiction**, cold start. **Evaluation**: online A/B — retention/dwell/engagement + guardrails (no harmful content, diversity, long-term retention vs short-term clicks). Interview line: *"Design a feed: candidates (follows + recommendation) → multi-objective value model (predict like/comment/dwell etc., fuse with a value function, using MMoE) → diversity re-ranking (MMR to prevent celebrity flooding) + freshness; architecture uses hybrid fan-out (push for ordinary users, pull for celebrities, merge at read) to solve celebrity write explosion; the key is multi-objective weights encoding values to prevent clickbait/addiction and the diversity-relevance tradeoff, judged by online A/B on long-term retention not short-term clicks."*


In [ ]:

# ============================================================
# ① 多目标价值模型:融合多个行为预测 / multi-objective value model: fuse multiple action predictions
# 中文:feed 不只优化点击。对每条候选, 模型预测多个行为的概率(点赞/评论/转发/时长/举报),
#      再用价值函数加权融合成一个排序分。权重编码产品价值观(奖励深度互动, 惩罚举报)。
# English: the feed optimizes more than clicks. For each candidate, the model predicts multiple action probabilities
#      (like/comment/share/dwell/report), fused by a value function. Weights encode product values.
# ============================================================
import numpy as np
np.random.seed(0)
candidates=[{"id":i, "p_like":np.random.rand(), "p_comment":np.random.rand()*0.3,
             "p_share":np.random.rand()*0.15, "p_dwell":np.random.rand(), "p_report":np.random.rand()*0.05}
            for i in range(6)]
# 价值函数:各行为加权(评论/转发权重高=鼓励深度互动; 举报是负权重=惩罚)/ value function: weighted sum
W={"p_like":1.0, "p_comment":3.0, "p_share":5.0, "p_dwell":2.0, "p_report":-20.0}   # 权重=产品价值观 / weights = product values
for c in candidates:
    c["value"]=round(sum(W[k]*c[k] for k in W), 3)                                   # 融合成排序分 / fuse into ranking score
print("多目标价值融合(权重: 转发5 > 评论3 > 时长2 > 点赞1, 举报 -20):")
for c in sorted(candidates, key=lambda x:-x["value"]):
    print(f"  帖子{c['id']}: like={c['p_like']:.2f} comment={c['p_comment']:.2f} share={c['p_share']:.2f} "
          f"report={c['p_report']:.3f} → 价值分 {c['value']:.2f}")
print("→ 若只优化点击(p_like)排序会不同; 多目标价值函数让 feed 鼓励深度互动、惩罚有害内容")


In [ ]:

# ============================================================
# ② 多样性重排(MMR):防止一个作者刷屏 / diversity re-ranking (MMR): prevent one author flooding
# 中文:Alice 是高产高质作者, 纯按价值分排, top6 全是她的帖子(体验单一)。MMR 在"相关性"和"作者多样性"间权衡。
# English: Alice is a prolific high-quality author; pure value-score order makes top6 all hers (monotonous). MMR trades off relevance vs author diversity.
# ============================================================
from collections import Counter
posts=([{"author":"Alice","rel":r} for r in [0.90,0.85,0.83,0.80,0.78,0.75,0.72,0.70]]   # 高产高分作者 / prolific high-scorer
      +[{"author":"Bob","rel":r}   for r in [0.71,0.66,0.62]]
      +[{"author":"Carol","rel":r} for r in [0.68,0.60,0.55]]
      +[{"author":"Dave","rel":r}  for r in [0.64,0.58]])
pure=sorted(posts, key=lambda p:-p["rel"])                                             # ✗ 纯相关性排序 / pure relevance
def mmr(posts, lam=0.6, k=6):                                                          # ✓ MMR 多样性重排 / MMR diversity
    selected, pool = [], posts[:]
    while pool and len(selected)<k:
        best, best_score = None, -1e9
        for p in pool:
            penalty = sum(1 for s in selected if s["author"]==p["author"])            # 与已选同作者的重复度 / same-author repetition
            score = lam*p["rel"] - (1-lam)*penalty                                     # 相关性 - 多样性惩罚 / relevance - diversity penalty
            if score>best_score: best_score, best = score, p
        selected.append(best); pool.remove(best)
    return selected
div=mmr(posts)
print("✗ 纯相关性 top6 作者:", [p["author"] for p in pure[:6]], "→", dict(Counter(p["author"] for p in pure[:6])), "← Alice 刷屏")
print("✓ MMR    top6 作者:", [p["author"] for p in div],       "→", dict(Counter(p["author"] for p in div)), "← 作者多样")
print(f"作者多样性: 纯相关性 {len(set(p['author'] for p in pure[:6]))} 种 → MMR {len(set(p['author'] for p in div))} 种")
print("→ MMR 牺牲一点点相关性换取多样性, 避免单一作者刷屏——feed 必备的重排")


In [ ]:

# ============================================================
# 可视化:fan-out 架构 + 多样性对比 / fan-out architecture + diversity comparison
# ============================================================
import matplotlib.pyplot as plt
fig,ax=plt.subplots(1,2,figsize=(14,5))
# ① 作者多样性对比 / author diversity
labels=["纯相关性排序","MMR 多样性重排"]
pure_c=Counter(p["author"] for p in pure[:6]); div_c=Counter(p["author"] for p in div)
authors=["Alice","Bob","Carol","Dave"]; cols=["#C44E52","#4C72B0","#55A868","#DD8452"]
bottom=[0,0]
for a,c in zip(authors,cols):
    vals=[pure_c.get(a,0), div_c.get(a,0)]
    ax[0].bar(labels, vals, bottom=bottom, label=a, color=c)
    bottom=[bottom[i]+vals[i] for i in range(2)]
ax[0].set_ylabel("top6 中各作者的帖子数"); ax[0].set_title("MMR:从 Alice 刷屏 → 4 位作者多样"); ax[0].legend(fontsize=8)
# ② fan-out 推 vs 拉 / push vs pull
ax[1].axis("off"); ax[1].set_title("架构:fan-out 推 vs 拉 vs 混合",fontsize=12,weight="bold")
rows=[("推(write)","发帖即写入所有粉丝收件箱","读快 ✓  大V写爆炸 ✗","#55A868"),
      ("拉(read)","读时实时合并所有关注者","写便宜 ✓  读慢 ✗","#DD8452"),
      ("混合(实践)","普通用户推 + 大V拉, 读时合并","两全, 工业标准 ✓","#4C72B0")]
for i,(m,how,tr,c) in enumerate(rows):
    y=0.72-i*0.24
    ax[1].add_patch(plt.Rectangle((0.05,y-0.02),0.9,0.2,fc=c,alpha=0.18,ec=c,transform=ax[1].transAxes))
    ax[1].text(0.08,y+0.13,m,fontsize=10,weight="bold",color=c,transform=ax[1].transAxes)
    ax[1].text(0.08,y+0.07,how,fontsize=8,transform=ax[1].transAxes)
    ax[1].text(0.08,y+0.01,tr,fontsize=8,style="italic",transform=ax[1].transAxes)
plt.tight_layout(); plt.savefig("/tmp/sd03_viz.png",dpi=80); plt.show()
print("左:MMR 把 Alice 刷屏的 feed 变多样; 右:fan-out 混合模式(普通用户推+大V拉)解决大V写爆炸")


**中文**:诚实解读:
**English**: Honest takeaways:

**中文**:
1. **feed 的灵魂是"多目标",而多目标的权重编码了产品的价值观和责任**:如果 feed 只优化一个指标,几乎必然走向极端——只优化点击率,系统会学会推标题党、震惊体;只优化观看时长,会推最让人上瘾、停不下来的内容(哪怕低质、极化)。这不是危言耸听,而是过去十年社交媒体真实发生的教训。**多目标价值模型是解药也是责任所在**:通过同时预测点赞、评论、转发、时长、关注、举报,并用价值函数加权(奖励深度互动、惩罚举报和有害内容),产品团队实际上在**用权重编码"我们想鼓励什么样的内容生态"**。我们的 demo 展示了同一批候选,只优化点击 vs 多目标价值函数会给出不同的排序——这个权重的设定是产品、伦理、商业的综合决策,是 feed 系统最有价值也最有争议的部分。
2. **多样性不是"锦上添花",而是 feed 的生存必需**:纯按分数排序有一个致命的局部最优——它会让**当下最高分的内容霸屏**。我们的 demo 里,Alice 作为高产高质作者,纯排序让她一人占满 top6,而真实产品里这意味着"你打开 app 全是一个人/一个话题",用户会疲劳、厌烦、流失。MMR 用一个简单的"相关性 − 重复度惩罚"权衡,牺牲一点点单条相关性,换来整体的多样和新鲜。这背后是一个深刻的道理:**feed 优化的不该是"单条内容的分数之和",而是"整个列表带给用户的总价值"**——而总价值天然包含多样性、新鲜度、不重复。这也是为什么"重排"是独立且必要的一层。
3. **诚实的难点:feed 的最大挑战是"短期指标 vs 长期健康"的根本冲突,和成瘾/极化的伦理**。①**短期 vs 长期**:能最大化今天点击/时长的排序,可能在慢慢损害长期留存(用户被标题党喂腻了、被极化内容激怒了,几个月后默默流失)。但长期留存极难在离线优化、也极难快速 A/B(要等很久)。这个"优化短期代理指标,却可能伤害长期真实目标"的困境,是 feed(乃至所有推荐)最深的难题,没有干净的技术解——需要精心设计护栏指标、长期 holdout 实验、和价值观驱动的权衡。②**成瘾与极化的伦理**:feed 的目标函数如果纯粹追求"参与度",会系统性地放大愤怒、极端、上瘾的内容(因为它们最"吸睛")——这是社交媒体被广泛批评的核心。负责任的 feed 设计必须主动把"内容质量、用户福祉、社会影响"纳入目标,而非只优化参与度。③**fan-out 的工程权衡**:推快但大 V 写爆炸,拉省但读慢,混合是标准答案——这是纯系统设计,但面试常考,要能清楚权衡。④**冷启动和反馈回路**同推荐(24.1)。**结论:设计 feed 在推荐漏斗上加了多目标价值模型(预测多行为、价值函数融合、权重编码价值观防标题党上瘾)和必须的多样性重排(MMR 防刷屏), 架构上用 fan-out 混合模式解决大V写爆炸; 但 feed 最深的挑战是短期参与度指标与长期用户健康的根本冲突, 以及成瘾/极化的伦理责任——顶尖的系统设计答案会主动提到护栏指标、长期实验和价值权衡, 而非只谈怎么最大化点击。**

**English**:
1. **The feed's soul is "multi-objective," and the weights encode the product's values and responsibility**: if a feed optimizes one metric, it almost inevitably goes to an extreme — optimize only click-through and the system learns to push clickbait and outrage; optimize only watch time and it pushes the most addictive, unstoppable content (even if low-quality, polarizing). Not alarmism but the real lesson of the past decade of social media. **The multi-objective value model is both the cure and where responsibility lies**: by predicting like, comment, share, dwell, follow, and report simultaneously and weighting them with a value function (rewarding deep engagement, penalizing reports and harmful content), the product team is actually **encoding "what kind of content ecosystem we want to encourage" via the weights**. Our demo shows the same candidates rank differently under click-only vs a multi-objective value function — setting these weights is a combined product/ethics/business decision, the most valuable and most contested part of a feed system.
2. **Diversity isn't "nice-to-have" but the feed's survival necessity**: pure score-order has a fatal local optimum — it lets **the currently highest-scoring content dominate**. In our demo, Alice as a prolific high-quality author fills the entire top-6 under pure ranking, which in a real product means "you open the app and it's all one person/topic," fatiguing and repelling users into churn. MMR uses a simple "relevance − repetition penalty" tradeoff, sacrificing a little per-item relevance for overall diversity and freshness. Behind this is a deep truth: **the feed should optimize not "the sum of individual content scores" but "the total value the whole list brings the user"** — and total value inherently includes diversity, freshness, non-repetition. This is why "re-ranking" is a separate and necessary layer.
3. **Honest difficulty: the feed's biggest challenge is the fundamental conflict of "short-term metrics vs long-term health," and the ethics of addiction/polarization**. ① **Short-term vs long-term**: the ranking that maximizes today's clicks/dwell may slowly erode long-term retention (users sick of clickbait, angered by polarizing content, quietly churning months later). But long-term retention is extremely hard to optimize offline and hard to A/B quickly (requires a long wait). This dilemma of "optimizing a short-term proxy while possibly harming the long-term true goal" is the feed's (and all recommendation's) deepest problem, with no clean technical fix — requiring carefully designed guardrail metrics, long-term holdout experiments, and value-driven tradeoffs. ② **The ethics of addiction and polarization**: a feed objective purely chasing "engagement" systematically amplifies angry, extreme, addictive content (because it's most "eye-catching") — the core of social media's widespread criticism. Responsible feed design must proactively include "content quality, user wellbeing, societal impact" in the objective, not just optimize engagement. ③ **Fan-out's engineering tradeoff**: push is fast but explodes writes for celebrities, pull is cheap but reads slowly, hybrid is the standard answer — pure system design, but frequently asked, so know the tradeoff clearly. ④ **Cold start and feedback loops** same as recommendation (24.1). **Conclusion: designing a feed adds a multi-objective value model (predict multiple actions, fuse with a value function, weights encoding values to prevent clickbait/addiction) and mandatory diversity re-ranking (MMR to prevent flooding) onto the recommendation funnel, with hybrid fan-out architecture solving celebrity write explosion; but the feed's deepest challenge is the fundamental conflict between short-term engagement metrics and long-term user health, plus the ethical responsibility of addiction/polarization — a top-tier system-design answer proactively raises guardrail metrics, long-term experiments, and value tradeoffs, not just how to maximize clicks.**

> 💼 **实战视角 / Practical angle**
> **中文**:feed 落地:①**候选**=关注/好友 + 推荐(涨活跃)+ 广告, 多路召回取并集;②**多目标价值模型**(MMoE/ESMM, 20.11)预测点赞/评论/转发/时长/关注/举报, 价值函数加权融合(权重是产品决策, 定期调);③**多样性重排** MMR/DPP(作者/话题/类型)+ 新鲜度 + 去重 + 不连续同类;④**架构** fan-out 混合(普通推、大V拉、读时合并);⑤**护栏**:不只看点击/时长, 监控长期留存、多样性、有害内容比例、用户举报——**用长期 holdout 实验**评估真实影响;⑥反馈回路/位置偏差同推荐。**答题**:先讲多目标价值模型和权重编码价值观, 再讲必须的多样性重排, 主动提 fan-out 架构和"短期指标 vs 长期健康"的权衡+伦理。面试金句:*"feed=候选(关注+推荐)→多目标价值模型(预测多行为, 价值函数融合, MMoE, 权重编码价值观防标题党)→多样性重排(MMR 防大V刷屏)+新鲜度; 架构 fan-out 混合(普通推大V拉)解决写爆炸; 最深难题是短期参与度与长期留存的冲突和成瘾极化伦理, 要护栏指标+长期实验+价值权衡, 而非只最大化点击。"*
> **English**: Feed in practice: ① **candidates** = follows/friends + recommendation (boost activity) + ads, multi-source unioned; ② **multi-objective value model** (MMoE/ESMM, 20.11) predicting like/comment/share/dwell/follow/report, fused by a value function (weights are a product decision, tuned periodically); ③ **diversity re-ranking** MMR/DPP (author/topic/type) + freshness + dedup + no consecutive same-type; ④ **architecture** hybrid fan-out (push ordinary, pull celebrities, merge at read); ⑤ **guardrails**: not just clicks/dwell, monitor long-term retention, diversity, harmful-content share, user reports — evaluate real impact with **long-term holdout experiments**; ⑥ feedback loops/position bias as in recommendation. **Answering**: cover the multi-objective value model and value-encoding weights first, then mandatory diversity re-ranking, proactively raise fan-out architecture and the "short-term metrics vs long-term health" tradeoff + ethics. Interview line: *"Feed = candidates (follows + recommendation) → multi-objective value model (predict multiple actions, fuse with a value function, MMoE, weights encoding values to prevent clickbait) → diversity re-ranking (MMR to prevent celebrity flooding) + freshness; hybrid fan-out architecture (push ordinary, pull celebrities) solves write explosion; the deepest challenge is the conflict between short-term engagement and long-term retention and the ethics of addiction/polarization, needing guardrail metrics + long-term experiments + value tradeoffs, not just maximizing clicks."*

---
### 小结 / Summary
- **中文**:feed=推荐漏斗 + ①多目标价值模型(预测多行为, 价值函数加权, 权重编码价值观)②必须的多样性重排(MMR 防刷屏)。
- **English**: Feed = recommendation funnel + ① multi-objective value model (predict multiple actions, weighted value function, weights encoding values) ② mandatory diversity re-ranking (MMR to prevent flooding).
- **中文**:架构 fan-out:推(读快写爆炸)/拉(写便宜读慢)/混合(普通推+大V拉, 工业标准)。
- **English**: Fan-out architecture: push (fast reads, write explosion) / pull (cheap writes, slow reads) / hybrid (push ordinary + pull celebrities, the industry standard).
- **中文**:最深难题=短期参与度指标 vs 长期用户健康的冲突 + 成瘾/极化伦理; 需护栏指标+长期实验+价值权衡。
- **English**: Deepest challenge = short-term engagement metrics vs long-term user health, plus addiction/polarization ethics; needs guardrail metrics + long-term experiments + value tradeoffs.
